Import libraries

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args
from skopt import gp_minimize

import datasets

In [12]:
file = "2023_Data_BiomassGasification_NED.xlsx"
data = pd.read_excel(file, sheet_name="Normalized Data")

data.head()

,feed_particle_size,feed_LHV,C,H,N,S,O,feed_ash,feed_moisture,feed_VM,...,other_bed,alumina,Y-alumina,calcium oxide,dolomite,olivine,silica,sand,lab,pilot
0,0.0681,0.236607,0.420376,0.458096,0.127396,0.042204,0.604911,0.467066,0.125441,0.196338,...,0,0,0,0,0,1,0,0,0,1
1,0.0681,0.236607,0.420376,0.458096,0.127396,0.042204,0.604911,0.467066,0.125441,0.196338,...,0,0,0,0,0,1,0,0,0,1
2,0.0681,0.236607,0.420376,0.458096,0.127396,0.042204,0.604911,0.467066,0.125441,0.196338,...,0,0,0,0,0,1,0,0,0,1
3,0.0681,0.236607,0.420376,0.458096,0.127396,0.042204,0.604911,0.467066,0.125441,0.196338,...,0,0,0,0,0,1,0,0,0,1
4,0.0681,0.236607,0.420376,0.458096,0.127396,0.042204,0.604911,0.467066,0.125441,0.196338,...,0,0,0,0,0,1,0,0,0,1


Datasets splitting

In [17]:
X = data.loc[:, ~data.columns.isin(['N2','H2','CO','CO2','CH4','C2Hn','gas_LHV','gas_tar','gas_yield','char_yield','CGE',
                                   'CCE', 'feed_cellulose','feed_hemicellulose','feed_lignin','residence_time','111.46',
                                   '205', 'atmospheric','slightly above atmospheric','slightly below atmospheric',
                                   'feed_LHV', 'feed_VM', 'feed_FC', 'other_feed_type', 'other_feed_shape',
                                   'continuous', 'other_gas', 'other_bed', 'dolomite'])]
Y = data["H2"]

X_train, X_remain, Y_train, Y_remain = train_test_split(X, Y, train_size=0.7, random_state=1)
X_validate, X_test, Y_validate, Y_test = train_test_split(X_remain, Y_remain, train_size=0.5, random_state=1)

Create evaluation function

In [31]:
def performance_evaluation(actual, predict):  # Model evaluation function

    # Computing R2 Score
    r2 = r2_score(actual, predict)

    # Computing Mean Square Error (MSE)
    mse = mean_squared_error(actual, predict)

    # Computing Mean Absolute Error (MAE)
    #mae = mean_absolute_error(actual, predict)

    # Show evaluation results
    print("------------------------------------------------------------------------")
    print(f"Model's performance evaluation score:")
    print(f"> R2: {r2:.4f}")
    print(f"> MSE: {mse:.4f}")
    #print(f"> MSE: {mae:.4f}")
    print("------------------------------------------------------------------------")

    eval_list = [r2, mse]

    return eval_list

Create Hyperparameter tuning function

In [37]:
# Create a list of the model's hyperparameter for tuning
search_space = list()
search_space.append(Integer(50, 500, name='n_estimators'))
search_space.append(Real(0.01, 1, prior='uniform', name='learning_rate'))
search_space.append(Integer(3, 20, name='max_depth'))
#search_space.append(Integer(1, 20, name='min_samples_leaf'))
#search_space.append(Integer(2, 20, name='min_samples_split'))
#search_space.append(Integer(1, 37, name='max_features'))
#search_space.append(Real(0.5, 1, prior='uniform', name='subsample'))


@use_named_args(search_space)
def Parameter_cv(**params):  # Function to return a minimizing score for hyperparameter tuning with cross-validation
    # Configure the model with specific hyperparameters
    GB_model = GradientBoostingRegressor()
    GB_model.set_params(**params)

    # Implementing repeated k-fold cross-validation
    cv = KFold(n_splits=k, shuffle=True, random_state=1)

    # Calculate 5-fold cross validation
    cv_score = cross_val_score(GB_model, X_train, Y_train, cv=cv, n_jobs=-1, scoring='r2')
    estimate = np.mean(cv_score)

    # Convert from a maximizing score to a minimizing score
    return 1.0 - estimate

Hyperparameter tuning

In [50]:
k = 5
result = gp_minimize(Parameter_cv, search_space)

# Summarizing finding:
print('Hyperparameters for 5-fold-cross-validation model')
print('Best Accuracy: %.3f' % (1 - result.fun))
print('Best Parameters: %s' % (result.x))

n_estimators, learning_rate, max_depth = result.x

Hyperparameters for 5-fold-cross-validation model
Best Accuracy: 0.902
Best Parameters: [500, 0.16803365678851498, 4]


Model config and fit

In [47]:
model = GradientBoostingRegressor(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth)
model.fit(X_train, Y_train)

Predict_train = model.predict(X_train)
df_predict_train = pd.DataFrame({'H2': Predict_train.flatten()})

performance_evaluation(Y_train, df_predict_train)

------------------------------------------------------------------------
Model's performance evaluation score:
> R2: 0.9994
> MSE: 0.0000
------------------------------------------------------------------------


[0.9994282010207879, 2.707045069386533e-05]

Model testing

In [49]:
Predict_test = model.predict(X_test)
df_predict_test = pd.DataFrame({'H2': Predict_test.flatten()})

performance_evaluation(Y_test, df_predict_test)

------------------------------------------------------------------------
Model's performance evaluation score:
> R2: 0.9471
> MSE: 0.0025
------------------------------------------------------------------------


[0.947131581084954, 0.0024737515576061086]